In [1]:
!pip install google-adk python-dotenv requests google-genai litellm google-cloud-modelarmor

In [ ]:
import os
from google.colab import userdata
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools.agent_tool import AgentTool
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.genai import types
from typing import Optional
from google.adk.tools import exit_loop
from google.adk.agents import SequentialAgent, LoopAgent, LlmAgent
import logging
import datetime
import sys

logger = logging.getLogger("WeatherAgent")
logger.setLevel(logging.INFO)

# Remove any existing handlers to prevent duplicates on cell re-runs
if logger.handlers:
    logger.handlers.clear()

# Force output to stdout so Colab displays it in the cell output
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(handler)
logger.propagate = False


# Required for Gemini and Geocoding
os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"

# OpenAI (if using GPT models)
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"

# Anthropic (if using Claude)
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_KEY"

GOOGLE_MAPS_API_KEY = "GOOGLE_API_KEY"

# Model Armor config — parsed from your template resource name
MODEL_ARMOR_PROJECT = "qwiklabs-gcp-04-a318f6cee29a"
MODEL_ARMOR_LOCATION = "us"
MODEL_ARMOR_TEMPLATE_ID = "Weather_ADK_Agent_Moderation_Template"
MODEL_ARMOR_TEMPLATE_NAME = (
    f"projects/{MODEL_ARMOR_PROJECT}/locations/{MODEL_ARMOR_LOCATION}"
    f"/templates/{MODEL_ARMOR_TEMPLATE_ID}"
)


In [6]:
from google.cloud import modelarmor_v1
from google.api_core.client_options import ClientOptions

# Model Armor uses a regional endpoint
armor_client = modelarmor_v1.ModelArmorClient(
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{MODEL_ARMOR_LOCATION}.rep.googleapis.com"
    )
)
print("Model Armor client initialized.")


Model Armor client initialized.


In [7]:
import requests
from typing import Dict

def get_coordinates(location_name: str) -> Dict[str, float]:
    """
    Converts a place name (city, address, etc.) into latitude and longitude.

    Args:
        location_name (str): The name of the city or place to geocode.

    Returns:
        dict: A dictionary with 'lat' and 'lon' keys.
    """
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location_name}&key={GOOGLE_MAPS_API_KEY}"
    response = requests.get(url).json()

    if response.get("status") == "OK":
        location = response["results"][0]["geometry"]["location"]
        return {"lat": location["lat"], "lon": location["lng"]}
    return {"error": "Location not found"}

def get_nws_weather(lat: float, lon: float) -> str:
    """
    Retrieves the current weather forecast from the National Weather Service (NWS).

    Args:
        lat (float): Latitude coordinate.
        lon (float): Longitude coordinate.
    """
    # NWS requires a User-Agent header (use your own email/app name)
    headers = {"User-Agent": "WeatherAlertAgent/1.0 (contact@example.com)"}

    # Step 1: Get grid point metadata from coordinates
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    res = requests.get(points_url, headers=headers).json()

    if "properties" not in res:
        return "Weather data unavailable for this location (NWS is US only)."

    # Step 2: Get the actual forecast from the grid endpoint provided in properties
    forecast_url = res["properties"]["forecast"]
    forecast_res = requests.get(forecast_url, headers=headers).json()

    # Get the current period's forecast summary
    current = forecast_res["properties"]["periods"][0]
    return (f"Forecast for {current['name']}: {current['detailedForecast']} "
            f"Temperature: {current['temperature']}°{current['temperatureUnit']}.")

In [8]:
def log_user_prompt(
    callback_context: CallbackContext,
    llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """
    Fires before every LLM call. Logs the latest user message.
    Returning None lets the agent continue normally.
    """
    if llm_request.contents:
        for content in reversed(llm_request.contents):
            if content.role == "user":
                user_text = " ".join(
                    part.text for part in content.parts
                    if hasattr(part, "text") and part.text
                )
                if user_text:
                    logger.info(
                        f"[Agent: {callback_context.agent_name}] "
                        f"[Session: {callback_context.session.id}] "   # <-- fix here
                        f"[Invocation: {callback_context.invocation_id}] "
                        f"User prompt: {user_text}"
                    )
                break

    return None


In [9]:
def validate_with_model_armor(
    callback_context: CallbackContext,
    llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """
    Fires before every LLM call.
    Sends the user prompt to Model Armor for sanitization.
    Blocks the LLM call if Model Armor flags the input.
    """
    if not llm_request.contents:
        return None

    # Extract the latest user message
    user_text = ""
    for content in reversed(llm_request.contents):
        if content.role == "user":
            user_text = " ".join(
                part.text for part in content.parts
                if hasattr(part, "text") and part.text
            ).strip()
            break

    if not user_text:
        return None

    try:
        # Call Model Armor sanitizeUserPrompt
        request = modelarmor_v1.SanitizeUserPromptRequest(
            name=MODEL_ARMOR_TEMPLATE_NAME,
            user_prompt_data=modelarmor_v1.DataItem(text=user_text),
        )
        response = armor_client.sanitize_user_prompt(request=request)
        result = response.sanitization_result

        # Check the verdict — MATCH_FOUND means the input was flagged
        match_state = result.filter_match_state
        is_flagged = (
            match_state == modelarmor_v1.FilterMatchState.MATCH_FOUND
        )

        if is_flagged:
            # Log which filters triggered
            logger.warning(
                f"[ModelArmor] Prompt blocked for input: '{user_text}' | "
                f"Match state: {match_state.name}"
            )
            return LlmResponse(
                content=types.Content(
                    role="model",
                    parts=[types.Part(
                        text=(
                            "Your request was flagged by our content policy "
                            "and cannot be processed. Please enter a valid US location."
                        )
                    )]
                )
            )

        logger.info(f"[ModelArmor] Prompt passed sanitization: '{user_text}'")
        return None  # allow the LLM call to proceed

    except Exception as e:
        # Fail open — if Model Armor is unreachable, log and continue
        logger.error(f"[ModelArmor] Sanitization error: {e}. Allowing request through.")
        return None


In [10]:
def before_model_handler(
    callback_context: CallbackContext,
    llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Chains validation first, then logging."""
    # Validate first — if it returns a response, stop here
    result = validate_with_model_armor(callback_context, llm_request)
    if result is not None:
        return result  # blocked — skip logging and LLM call

    # If valid, log and continue
    return log_user_prompt(callback_context, llm_request)


In [11]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.agents import RunConfig

def extract_next_search_query(
    callback_context: CallbackContext,
) -> None:
    """
    After validator_agent runs, check if its output contains a
    NEXT_SEARCH_QUERY directive and update session state accordingly.
    """
    final_plan = callback_context.state.get("final_plan", "")

    if "NEXT_SEARCH_QUERY:" in final_plan:
        # Parse the new query from the validator's response
        for line in final_plan.splitlines():
            if line.strip().startswith("NEXT_SEARCH_QUERY:"):
                new_query = line.split("NEXT_SEARCH_QUERY:", 1)[1].strip()
                callback_context.state["venue_search_query"] = new_query
                logger.info(f"[Validator] Refined search query: '{new_query}'")
                break


In [12]:
def log_model_response(
    callback_context: CallbackContext,
    llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """
    Fires after every LLM call. Logs the model's response text.
    Returning None lets the agent continue normally with the original response.
    """
    if llm_response.content and llm_response.content.parts:
        response_text = " ".join(
            part.text for part in llm_response.content.parts
            if hasattr(part, "text") and part.text
        )
        if response_text:
            logger.info(
                f"[Agent: {callback_context.agent_name}] "
                f"[Session: {callback_context.session.id}] "
                f"[Invocation: {callback_context.invocation_id}] "
                f"Model response: {response_text}"
            )
        else:
            # Model made a tool call instead of returning text
            func_calls = [
                part.function_call.name for part in llm_response.content.parts
                if hasattr(part, "function_call") and part.function_call
            ]
            if func_calls:
                logger.info(
                    f"[Agent: {callback_context.agent_name}] "
                    f"[Session: {callback_context.session.id}] "
                    f"[Invocation: {callback_context.invocation_id}] "
                    f"Model called tools: {func_calls}"
                )

    return None  # MUST return None to pass original response through unchanged


In [13]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

def create_weather_agent(model_choice="gemini-flash-lite-latest"):
    """
    Creates a weather agent. Pass a Gemini model name as a string,
    or a LiteLlm() object for third-party models.
    """
    agent_instructions = """
    You are a helpful weather analyst.
    1. Use 'get_coordinates' to find the lat/lon of a user's location.
    2. Use 'get_nws_weather' with those coordinates to get the forecast.
    3. Provide a clear summary including:
       - Temperature range
       - Precipitation (rain, snow, storms)
       - Wind conditions
       - Overall outdoor suitability (safe / risky / dangerous)
    Keep the answer concise — it will be used by a planning system downstream.
    """

    return LlmAgent(
        name="weather_agent",
        model=model_choice,        # accepts string for Gemini, or LiteLlm() for others
        instruction=agent_instructions,
        tools=[get_coordinates, get_nws_weather],
        output_key="weather_report",
        before_model_callback=before_model_handler,
        after_model_callback=log_model_response,
    )


In [14]:
from google.adk.agents import LlmAgent
from google.adk.tools import google_search

def create_venue_search_agent(model_choice="gemini-flash-lite-latest"):
    """
    Search agent that specializes in web search using the built-in google_search tool.
    """
    search_instructions = """
    You are a venue search specialist.

    Search for venues using this query: {venue_search_query}

    Use the 'google_search' tool to find 3 specific venue options.
    For each venue provide:
    - Name
    - Indoor or Outdoor
    - A brief description

    Keep results short and structured.
    """

    return LlmAgent(
        name="search_agent",
        model=model_choice,
        instruction=search_instructions,
        tools=[google_search],
        output_key="venue_results",
    )


In [15]:
def create_validator_agent(model_choice="gemini-flash-lite-latest"):
    validator_instructions = """
    You are a travel safety validator.

    You have:
    - Weather report: {weather_report}
    - Venue results: {venue_results}
    - User's original request: {user_request}

    Your job:
    1. Check if the venues are appropriate given the weather:
       - If weather is SAFE (no rain, no storms, mild conditions):
         ANY venue (indoor or outdoor) is acceptable.
       - If weather is RISKY or DANGEROUS (rain, storms, high winds):
         Only INDOOR venues are acceptable. Outdoor venues must be rejected.

    2. Decision:
       A. If venues ARE appropriate for the weather:
          - Write a friendly final travel plan combining the weather summary
            and the recommended venue(s).
          - Save it as your response and call the 'exit_loop' tool.

       B. If venues are NOT appropriate (e.g., outdoor venues in bad weather):
          - Identify what went wrong (e.g., "venues are outdoor but it will rain").
          - Rewrite a better search query (e.g., "indoor event venues Austin Texas").
          - Save the new search query as {venue_search_query} by including it
            in your response in this exact format:
            NEXT_SEARCH_QUERY: <your new query here>
          - Do NOT call exit_loop. The loop will retry with the new query.
    """
    return LlmAgent(
        name="validator_agent",
        model=model_choice,
        instruction=validator_instructions,
        tools=[exit_loop],
        output_key="final_plan",
        after_agent_callback=extract_next_search_query,
    )


In [16]:
def create_query_initializer_agent(model_choice="gemini-flash-lite-latest"):
    init_instructions = """
    You are a search query builder.
    The user's request is: {user_request}

    Generate a first venue search query based on the user's request.
    Examples:
    - "outdoor rooftop bars Austin Texas"
    - "event venues for outdoor party Chicago"

    Respond with ONLY the search query, nothing else.
    """
    return LlmAgent(
        name="query_initializer_agent",
        model=model_choice,
        instruction=init_instructions,
        output_key="venue_search_query",       # seeds the initial search query
    )


In [17]:
def create_travel_planner_pipeline(model_choice="gemini-flash-lite-latest"):

    # Sub-agents
    weather_agent    = create_weather_agent(model_choice)
    query_init_agent = create_query_initializer_agent(model_choice)
    venue_search     = create_venue_search_agent(model_choice)
    validator        = create_validator_agent(model_choice)

    # Loop: search → validate → refine if needed
    venue_finder_loop = LoopAgent(
        name="VenueFinderLoop",
        sub_agents=[venue_search, validator],
        max_iterations=5,                      # stop after 5 tries max
    )

    # Full pipeline: weather → init query → loop
    pipeline = SequentialAgent(
        name="TravelPlannerPipeline",
        sub_agents=[
            weather_agent,                     # Step 1: get forecast
            query_init_agent,                  # Step 2: build first search query
            venue_finder_loop,                 # Step 3: search + validate loop
        ],
        description="Plans safe travel by checking weather before finding venues.",
    )

    return pipeline


In [18]:
async def chat_once(user_id: str, session_id: str, message: str) -> str:
    """
    Send one message to the agent and return its final response text.
    For pipeline (SequentialAgent/LoopAgent) roots, falls back to session state.
    """
    user_content = types.Content(
        parts=[types.Part(text=message)],
        role="user",
    )

    final_text = ""

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=user_content,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    final_text = part.text
                    break

    # Fallback: if pipeline root didn't emit a final response event,
    # read the result directly from session state
    if not final_text:
        session = await session_service.get_session(
            app_name=APP_NAME,
            user_id=user_id,
            session_id=session_id,
        )
        # Try keys in priority order
        final_text = (
            session.state.get("final_plan") or
            session.state.get("venue_results") or
            session.state.get("weather_report") or
            "Pipeline completed but no output was found in session state."
        )

    return final_text


In [19]:
APP_NAME = "travel_planner_app"
USER_ID = "demo_user"
SESSION_ID = "session_travel_1"

session_service = InMemorySessionService()
pipeline = create_travel_planner_pipeline(model_choice="gemini-flash-lite-latest")

runner = Runner(
    agent=pipeline,
    app_name=APP_NAME,
    session_service=session_service,
)

async def setup():
    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )
    print(f"Session created: {session.id}")

await setup()


Session created: session_travel_1


In [24]:
async def run_travel_planner(location: str, event_description: str):
    user_request = f"{event_description} in {location}"
    import uuid
    session_id = f"session_travel_{uuid.uuid4().hex[:8]}"

    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
        state={
            "user_request": user_request,
            "venue_search_query": f"outdoor event venues {location}",
        }
    )
    print(f"Session created: {session.id}")

    response = await chat_once(
        user_id=USER_ID,
        session_id=session_id,
        message=f"Plan a safe outdoor event: {user_request}",
    )

    # Show all relevant state keys for debugging
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    print("\n--- Session State ---")
    for key in ["weather_report", "venue_search_query", "venue_results", "final_plan"]:
        val = session.state.get(key, "NOT SET")
        print(f"\n[{key}]:\n{val}")

    print("\n--- Final Response ---")
    print(response)

await run_travel_planner(
    location="Austin, Texas",
    event_description="rooftop birthday party this Friday afternoon",
)


Session created: session_travel_62bdeca3
2026-03-06 02:40:21,235 [INFO] [ModelArmor] Prompt passed sanitization: 'Plan a safe outdoor event: rooftop birthday party this Friday afternoon in Austin, Texas'
2026-03-06 02:40:21,236 [INFO] [Agent: weather_agent] [Session: session_travel_62bdeca3] [Invocation: e-247264bb-d237-4389-bb49-1dfd0a6ac12d] User prompt: Plan a safe outdoor event: rooftop birthday party this Friday afternoon in Austin, Texas
2026-03-06 02:40:21,779 [INFO] [Agent: weather_agent] [Session: session_travel_62bdeca3] [Invocation: e-247264bb-d237-4389-bb49-1dfd0a6ac12d] Model called tools: ['get_coordinates']
2026-03-06 02:40:22,348 [INFO] [Agent: weather_agent] [Session: session_travel_62bdeca3] [Invocation: e-247264bb-d237-4389-bb49-1dfd0a6ac12d] Model called tools: ['get_nws_weather']
2026-03-06 02:40:24,360 [INFO] [Agent: weather_agent] [Session: session_travel_62bdeca3] [Invocation: e-247264bb-d237-4389-bb49-1dfd0a6ac12d] Model response: The user is asking for a plan 